# 09 - Pass 4 analysis and pair checks (CPU only, no GPU)

Runs after notebook 08. It trains no BERT model, so it needs **no GPU**: on Kaggle set **Accelerator: None** (CPU sessions do not use GPU quota). About 5-15 minutes. Code: `src/gainsplit/analysis_pass4.py` and `src/gainsplit/pair_checks.py`.

**Inputs (right-hand panel, Add Input):**
1. **`gainsplit-processed`** (the version with the pair files, same as notebook 08).
2. **The output of notebook 08** (search for your notebook 08 under "Notebook Output Files"). It holds `results/pass4/`.

**Internet: on** (fetches the code and the Pass 1-3 baseline runs from GitHub).

Every test and contrast was fixed before the runs (vault: Experiment Plan, "Pass 4 plan"). Scores are in points (macro-F1 0.882 = 88.2). All tables are saved to `/kaggle/working/analysis_pass4/`; download `analysis_pass4.zip` at the end.

| Part | Question |
|---|---|
| A. Pair checks | Can a word model tell generated from real headlines? Do the generated sides read as intended to a model trained on real headlines? Do outlet words or length predict the label in the pairs? |
| A5. Shortcut-pattern hunting | Which words/phrases and surface features does the AI actually use to write "sarcastic" vs "serious", beyond length and outlet cues? |
| B. Re-run check | Did the new code reproduce the Pass 1 / 2b runs exactly? |
| C. Contrasts | Do the pairs help (A10, A11)? Does pairing matter (A12)? Does the judge filter matter (A13 vs A14)? Pairs only (A15)? |
| D. Shortcut, transfer, explanations | Do pair-trained models lean less on outlet words, transfer better to iSarcasmEval, and put less attribution on cue words? |

## 1. Get the code and find the inputs

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    REPO_DIR = "/tmp/GainSplit"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Ariffurrhmn/GainSplit.git", REPO_DIR], check=True)
    INPUT = Path("/kaggle/input")
    OUT_DIR = Path("/kaggle/working/analysis_pass4")
else:  # local run from the repo: results/pass4 must hold the unzipped notebook 08 output
    REPO_DIR = str(Path.cwd().parent) if Path.cwd().name == "experiments" else str(Path.cwd())
    INPUT = Path(REPO_DIR) / "data" / "kaggle_upload"   # the folder prepared for upload
    OUT_DIR = Path(REPO_DIR) / "results" / "analysis_pass4"
sys.path.insert(0, f"{REPO_DIR}/src")
os.environ["GAINSPLIT_INPUT"] = str(INPUT)
import pandas as pd
pd.set_option("display.width", 200)
from gainsplit.data import find_data_dir, verify, load_pairs, load_cue_words
from gainsplit import analysis as an, analysis_pass4 as a4, pair_checks as pc

DATA_DIR = find_data_dir(str(INPUT))
verify(DATA_DIR)
R = Path(REPO_DIR) / "results"
BASELINE_DIRS = [R / p for p in ("pass1", "pass2a", "pass2b", "pass3")]          # committed to the repo
PASS4_DIRS = sorted({p.parent for p in Path(INPUT if ON_KAGGLE else R).rglob("runs")
                     if p.is_dir() and p.parent.name in ("pass4", "pass4_full100")})
assert PASS4_DIRS, "no results/pass4/runs found: add the output of notebook 08 as an input"
RESULTS_DIRS = BASELINE_DIRS + PASS4_DIRS
OUT_DIR.mkdir(parents=True, exist_ok=True)
N_BOOT, RNG_SEED = 10000, 0
print("data:", DATA_DIR)
print("results:", [str(p) for p in RESULTS_DIRS])

## A. Pair checks (word models only, a few minutes)
**A1. Style probe.** ROC-AUC of a word model (TF-IDF + logistic regression, 5-fold cross-validation) that separates generated from real training headlines (0.5 = cannot tell). A high value is expected and is not itself a problem: within a pair both sides are generated, so "generated" cannot predict the label. The paper reports how visible LLM style is.

**A2. Real-trained reader.** A word model trained on the real labelled training headlines scores every generated headline: AUC with the intended label as truth (real dev as reference row), mean P(sarcastic) per side, share of pairs whose sarcastic side scores higher (0.5 = no difference). No LLM judge involved.

**A3. Shortcuts.** AUC of the 200 outlet cue words alone, real vs generated (should drop towards 0.5 on the pairs; optimistic on real data because the words were chosen there), and AUC of headline length alone. Generated sarcastic sides are longer: if length predicts the label much better in the pairs than in real data, a pair-trained model could learn "long = sarcastic". Report it either way.

**A4. iSarcasmEval length shortcut.** The pair arms transfer well to iSarcasmEval (AUC 72-76 vs ~50 for A1/A7, see section D) while generated sarcastic sides are longer (A3 above). This checks whether word count alone already separates the real iSarcasmEval test labels, using the same items the models were scored on. If this AUC is itself high, the transfer gain may be partly a length shortcut rather than sarcasm.

In [ ]:
read = lambda n: pd.read_csv(DATA_DIR / n, keep_default_na=False)
train, dev = read("headlines_train.csv"), read("headlines_dev.csv")
pairs, info = load_pairs("unfiltered", DATA_DIR, check_against=DATA_DIR)
print(info)
style = pc.style_probe(train, pairs)
reader, _ = pc.real_reader(train, dev, pairs)
cues = pc.cue_predictiveness(train, pairs, load_cue_words(DATA_DIR))
lengths = pc.length_table(train, pairs)
isarc_test = read("isarcasm_test.csv")
isarc_shortcut = pc.isarc_length_shortcut(isarc_test, pairs, train)
for name, t in (("pairs_style_probe", style), ("pairs_real_reader", reader),
                ("pairs_cue_words", cues), ("pairs_length", lengths),
                ("isarc_length_shortcut", isarc_shortcut)):
    an.save_table(t, OUT_DIR, name, default_decimals=3)
    print(name); display(t.round(4))

## A5. Shortcut-pattern hunting (which words, not just length)

A3 above checks two named features chosen by hand (outlet cue words, length). This section looks
for the patterns instead of guessing them one at a time, so the next round of generation can be
told explicitly what to avoid.

**A5.1 Word/phrase table (`shortcut_word_table`).** One interpretable model (TF-IDF unigrams and
bigrams, logistic regression, same recipe as the style probe) is fit on all the generated pairs to
predict the AI's own sarcastic/serious label. Its top 30 words/phrases each side are the actual
vocabulary of "AI-sarcastic-speak" and "AI-serious-speak" — read `cv_auc` first: a high
cross-validated AUC means the model (and so the list below it) is picking up a real, learnable
pattern, not noise. `real_rate_sarcastic` / `real_rate_serious` say how common each word is in the
*real* training headlines by class: a word near 0 in both real columns is a pure AI tic (safe to
suppress in the next generation prompt or filter); a word with a real gap in the same direction is
a signal the pairs share with real sarcasm, not an artifact.

**A5.2 Named surface features beyond length (`shortcut_feature_table`).** AUC of each single
feature alone (word count, character count, exclamation/question marks, ALL-CAPS word count, and a
short filler-word list: thus / therefore / clearly / obviously / literally / honestly / indeed /
surely) against the generated label, with the same feature's AUC on the real training label next to
it for comparison. A feature with high `generated_auc` and near-chance `real_auc` is exactly the
kind of shortcut A5.1 is meant to catch from the word side; this is the explicit, named-feature
cross-check.

**Reading this section:** anything here with a high AUC on the generated side and a near-chance AUC
on the real side is an AI writing tic, not a real sarcasm signal, and a candidate for the next
generation round's "avoid" list ([[Research Ideas]] "After Pass 4").

In [ ]:
word_table = pc.shortcut_word_table(pairs, train, top_k=30)
print("word table cv_auc (whole-model):", round(word_table.attrs["cv_auc"], 4))
feature_table = pc.shortcut_feature_table(pairs, train)
for name, t in (("pairs_shortcut_words", word_table), ("pairs_shortcut_features", feature_table)):
    an.save_table(t, OUT_DIR, name, default_decimals=4)
    print(name); display(t)

## B. Load the runs and check the re-runs (read this first)
The A1 re-runs at 10% (tag `explain`) and the A1 / A7 re-runs at 1% (tag `mask`) repeat Pass 1 / 2b runs with the new code. Their test scores and probabilities must be identical (`ok` = True). If not, the new code changed the old training path and the contrasts against old baselines are not clean: stop and report.

In [ ]:
runs = an.load_runs(*RESULTS_DIRS)
print(len(runs), "runs loaded")
rr = a4.check_reruns(runs[runs["budget"].isin([1, 10])])
display(rr)
print("re-runs identical:", bool(len(rr)) and bool(rr["ok"].all()))
display(a4.summary(runs))

## C. Main contrasts (paired bootstrap, 10,000 resamples)
Difference in test macro-F1 (points), 95% interval, two-sided p, and Holm-adjusted p over all Pass 4 contrasts (robustness check, as in Passes 1-3).
- **A10-A1 / A11-A7 / A11-A1 / A11-A6:** do the pairs help, alone and on top of FGM, and against the best 1% baseline?
- **A10-A12 pairing:** near 0 means "generated text helps", not "counterfactual pairs help".
- **A13-A14 judge filter:** the judge's choice at equal size and model mix.
- **A15-A1:** pairs only (no human labels) vs 10% human labels.

In [ ]:
con = a4.contrast_table(runs, N_BOOT, RNG_SEED)
con.round(4)

## D. Outlet shortcut, transfer and explanations
- **Gap contrasts:** shortcut gap (random-mask minus cue-mask macro-F1) of a pair arm minus its baseline's. Negative = leans less on outlet words.
- **AUC contrasts:** iSarcasmEval ROC-AUC difference (points). In-domain reference from Pass 3: 65.8.
- **Explanations:** share of Integrated Gradients attribution on cue words, A10 vs A1 at 10% (same 500 test headlines for every model and seed). `median_rel_completeness_gap` should be a few % at most.
- **iSarcasmEval length shortcut:** bootstrap AUC of word count alone on the actual iSarcasmEval test labels and items (same ids the models were scored on), for comparison with the AUCs just above. If this AUC is itself high, the transfer gain may be a length shortcut; report both numbers together, do not quote the transfer AUC alone.

In [ ]:
expl = a4.load_explain(runs)
display(a4.gap_contrasts(runs, N_BOOT, RNG_SEED).round(4))
display(a4.auc_table(runs, N_BOOT, RNG_SEED).round(4))
display(a4.explain_table(expl).round(4))
display(a4.explain_contrast_table(expl, N_BOOT, RNG_SEED).round(4))
print(a4.isarc_length_auc(isarc_test, runs, N_BOOT, RNG_SEED))

## E. Save every table and zip them

In [ ]:
tables = a4.run_all(RESULTS_DIRS, OUT_DIR, N_BOOT, RNG_SEED, isarc_test=isarc_test)
print("tables:", [k for k, v in tables.items() if len(v)])
if ON_KAGGLE:
    print(shutil.make_archive("/kaggle/working/analysis_pass4", "zip", OUT_DIR))